# COMET offline — 5 cấu hình weight

- không internet
- dùng checkpoint + XLM-R cache đã attach
- 1688: COMET-da; Tiki: COMET-QE


In [ ]:
SUITE = 'B'
import os, sys, subprocess, zipfile, json, time
from pathlib import Path

# cài wheel offline trước khi import comet
wheel_hits = sorted(Path('/kaggle/input').rglob('unbabel_comet-*.whl'))
print('wheel hits', [str(x) for x in wheel_hits[:2]])
WHEELS = wheel_hits[0].parent if wheel_hits else None
if WHEELS is not None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                    '--find-links', str(WHEELS), '--no-deps',
                    'unbabel-comet==2.2.7', 'pytorch-lightning==2.6.6',
                    'torchmetrics==1.9.0', 'lightning-utilities==0.15.3',
                    'entmax==1.3', 'sacrebleu==2.6.0',
                    'jsonargparse', 'pyyaml'], check=False)
print('wheels done')


In [ ]:
# tìm checkpoint + HF cache; đặt biến môi trường offline trước khi import HF
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/comet_data'); WORK.mkdir(parents=True, exist_ok=True)
print('input dirs', [p.name for p in INPUT.iterdir()])
part_hits = sorted(INPUT.rglob('wmt22-da.ckpt.part.aa'))
single_hits = sorted(INPUT.rglob('wmt22-da.ckpt'))
if part_hits:
    FILES = part_hits[0].parent
    for name in ['wmt22-da','wmt20-qe']:
        d = WORK / f'checkpoints/{name}/checkpoints'; d.mkdir(parents=True, exist_ok=True)
        target = d / 'model.ckpt'
        if not target.exists():
            parts = sorted(FILES.glob(f'{name}.ckpt.part.*'))
            with open(target, 'wb') as out:
                for part in parts:
                    with open(part, 'rb') as fh: out.write(fh.read())
        hp = WORK / f'checkpoints/{name}/hparams.yaml'
        if not hp.exists(): hp.symlink_to(FILES / f'{name}.hparams.yaml')
elif single_hits:
    FILES = single_hits[0].parent
    for name in ['wmt22-da','wmt20-qe']:
        d = WORK / f'checkpoints/{name}/checkpoints'; d.mkdir(parents=True, exist_ok=True)
        target = d / 'model.ckpt'
        if not target.exists(): target.symlink_to(FILES / f'{name}.ckpt')
        hp = WORK / f'checkpoints/{name}/hparams.yaml'
        if not hp.exists(): hp.symlink_to(FILES / f'{name}.hparams.yaml')
else:
    raise SystemExit('không thấy checkpoint COMET trong /kaggle/input')
if not (WORK / 'hf').exists():
    zips = sorted(INPUT.rglob('xlm-roberta-large.zip'))
    hf_dirs = sorted(INPUT.rglob('xlm-roberta-large/hf'))
    if zips:
        with zipfile.ZipFile(zips[0]) as zf: zf.extractall(WORK)
    elif hf_dirs:
        import shutil; shutil.copytree(hf_dirs[0], WORK / 'hf')
    else:
        raise SystemExit('không thấy XLM-R cache')
COMET_BASE = WORK
os.environ['HF_HOME'] = str(WORK / 'hf')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
print('comet base', COMET_BASE)
print('checkpoint MB', round((COMET_BASE/'checkpoints/wmt22-da/checkpoints/model.ckpt').stat().st_size/1e6,1))
import torch, pandas as pd, numpy as np
from comet.encoders.xlmr import XLMREncoder
def _patched_forward(self, input_ids, attention_mask, **kwargs):
    outputs = self.model(input_ids=input_ids, attention_mask=attention_mask,
                         output_hidden_states=True, return_dict=False)
    if len(outputs) == 3:
        last_hidden_states, _, all_layers = outputs
    else:
        last_hidden_states, all_layers = outputs
    return {'sentemb': last_hidden_states[:, 0, :],
            'wordemb': last_hidden_states, 'all_layers': all_layers,
            'attention_mask': attention_mask}
XLMREncoder.forward = _patched_forward
print('patched xlmr runtime')
from comet import load_from_checkpoint
print('gpu', torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
t0=time.time()
da = load_from_checkpoint(str(COMET_BASE/'checkpoints/wmt22-da/checkpoints/model.ckpt'), local_files_only=True)
qe = load_from_checkpoint(str(COMET_BASE/'checkpoints/wmt20-qe/checkpoints/model.ckpt'), local_files_only=True)
print('loaded in', round(time.time()-t0,1), 's')


In [ ]:
hyp_files = sorted(p for p in Path('/kaggle/input').rglob('hyp_*.parquet')
                   if f'_{SUITE}_' in p.name)
print('hyp files', [str(x) for x in hyp_files])
assert hyp_files, 'không thấy hypothesis'
OUT = Path('/kaggle/working/scores'); OUT.mkdir(parents=True, exist_ok=True)
metrics = {}
for path in hyp_files:
    df = pd.read_parquet(path).drop_duplicates('eval_id')
    config = str(df['config'].iloc[0]); seed = int(df['seed'].iloc[0])
    for dataset in ['1688','tiki']:
        part = df[df['dataset']==dataset].copy()
        part = part[part['hypothesis'].astype(str).str.strip().ne('')]
        if dataset=='1688':
            part = part[part['reference'].astype(str).str.strip().ne('')]
            data=[{'src':str(s),'mt':str(m),'ref':str(r)} for s,m,r in zip(part.source,part.hypothesis,part.reference)]
            out=da.predict(data, batch_size=8, gpus=1)
            try:
                import sacrebleu
                bleu=sacrebleu.corpus_bleu(part.hypothesis.tolist(),[part.reference.tolist()],tokenize='zh').score
                chrf=sacrebleu.corpus_chrf(part.hypothesis.tolist(),[part.reference.tolist()]).score/100
                extra={'bleu':round(float(bleu),4),'chrf':round(float(chrf),4)}
            except Exception as e:
                print('skip bleu/chrf', e); extra={}
        else:
            data=[{'src':str(s),'mt':str(m)} for s,m in zip(part.source,part.hypothesis)]
            out=qe.predict(data, batch_size=8, gpus=1)
            extra={}
        part['score']=np.asarray(out.scores,dtype=float)
        part['config']=config; part['seed']=seed; part['dataset']=dataset
        part[['eval_id','pair_type','score','config','seed','dataset']].to_parquet(OUT/f'scores_{config}_seed{seed}_{dataset}.parquet',index=False)
        key=f'{config}::{seed}::{dataset}'
        metrics[key]={'system_score':round(float(out.system_score),4),'n':int(len(part)),**extra}
        print(key, metrics[key], flush=True)
(OUT/'comet_scores.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(metrics,ensure_ascii=False,indent=2))
